# Activity 10 — Field validation and accuracy assessment

**Learning objective:** Use observations that were not used to train the model to test whether the classification is reliable.

This activity connects the Tuesday validation survey back to the Friday analysis.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

## 1. Load independent validation points

The validation layer should contain a numeric reference-class field.  
Change `REFERENCE_FIELD` to your actual field name.

In [ ]:
VALIDATION_PATH = "Validation_Data/validation_points.geojson"
REFERENCE_FIELD = "randomforest"

validation = gpd.read_file(VALIDATION_PATH)

print(validation.head())
print("\nReference class counts:")
print(validation[REFERENCE_FIELD].value_counts().sort_index())

## 2. Sample the predicted raster at validation locations

In [ ]:
PREDICTION_RASTER = "Results/Efate_Invasive_Prediction.tif"

with rasterio.open(PREDICTION_RASTER) as src:
    validation_raster_crs = validation.to_crs(src.crs)

    coords = [
        (geom.x, geom.y)
        for geom in validation_raster_crs.geometry
    ]

    predicted_samples = np.array(
        [sample[0] for sample in src.sample(coords)]
    )

validation_results = validation_raster_crs.copy()
validation_results["predicted_class"] = predicted_samples

validation_results[
    [REFERENCE_FIELD, "predicted_class"]
].head()

## 3. Remove validation locations that fall on NoData

In [ ]:
valid = (
    np.isfinite(validation_results[REFERENCE_FIELD]) &
    np.isfinite(validation_results["predicted_class"])
)

assessment = validation_results.loc[valid].copy()

y_true = assessment[REFERENCE_FIELD].astype(int)
y_pred = assessment["predicted_class"].astype(int)

print("Validation points used:", len(assessment))

## 4. Calculate accuracy

In [ ]:
print("Overall accuracy:", accuracy_score(y_true, y_pred))
print()
print(classification_report(y_true, y_pred))

In [ ]:
labels = sorted(
    set(y_true.unique()).union(set(y_pred.unique()))
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(ax=ax)
plt.title("Independent Field Validation Confusion Matrix")
plt.show()

## 5. Interpretation questions

Discuss as a group:

1. Which invasive class has the highest recall?
2. Which class is most often confused with another vegetation class?
3. Are there areas of obvious overestimation?
4. Are there areas where the model missed confirmed field observations?
5. Would additional flowering-period imagery improve separability?
6. What additional training or validation observations are required?
7. Is the model accurate enough for the intended management use?

**Final message:** A model prediction is not ground truth. Independent field validation is essential.